# Testing PF-GIM on GPT-2 (Greater-Than Task)

Compares PF-GIM (Proximity-Filtered GIM) against GIM and EAP-IG on the greater-than circuit.

In [1]:
from functools import partial

import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import PreTrainedTokenizer
from transformer_lens import HookedTransformer

from eap.graph import Graph
from eap.evaluate import evaluate_graph, evaluate_baseline
from eap.attribute import attribute

KeyboardInterrupt: 

## Dataset and Metrics

In [ ]:
def collate_EAP(xs):
    clean, corrupted, labels = zip(*xs)
    clean = list(clean)
    corrupted = list(corrupted)
    return clean, corrupted, labels

class EAPDataset(Dataset):
    def __init__(self, filepath):
        self.df = pd.read_csv(filepath)

    def __len__(self):
        return len(self.df)

    def shuffle(self):
        self.df = self.df.sample(frac=1)

    def head(self, n: int):
        self.df = self.df.head(n)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        return row['clean'], row['corrupted'], row['label']

    def to_dataloader(self, batch_size: int):
        return DataLoader(self, batch_size=batch_size, collate_fn=collate_EAP)

def get_logit_positions(logits: torch.Tensor, input_length: torch.Tensor):
    batch_size = logits.size(0)
    idx = torch.arange(batch_size, device=logits.device)
    logits = logits[idx, input_length - 1]
    return logits

def get_prob_diff(tokenizer: PreTrainedTokenizer):
    year_indices = torch.tensor([tokenizer(f'{year:02d}').input_ids[0] for year in range(100)])

    def prob_diff(logits, clean_logits, input_length, labels, mean=True, loss=False):
        logits = get_logit_positions(logits, input_length)
        probs = torch.softmax(logits, dim=-1)[:, year_indices]
        results = []
        for prob, year in zip(probs, labels):
            results.append(prob[year + 1:].sum() - prob[:year + 1].sum())
        results = torch.stack(results)
        if loss:
            results = -results
        if mean:
            results = results.mean()
        return results
    return prob_diff

## Load Model and Data

In [ ]:
model_name = 'pythia-70m'
device = 'cuda'

model = HookedTransformer.from_pretrained(model_name, device=device)
model.cfg.use_split_qkv_input = True
model.cfg.use_attn_result = True
model.cfg.use_hook_mlp_in = True

metric = get_prob_diff(model.tokenizer)

dataset = EAPDataset('greater_than_data.csv')
dataloader = dataset.to_dataloader(batch_size=120)

Loaded pretrained model gpt2-small into HookedTransformer


## Baseline: Full Model Performance

In [ ]:
baseline = evaluate_baseline(model, dataloader, metric, quiet=False)
baseline_mean = baseline.mean().item()
print(f"Full model prob_diff: {baseline_mean:.4f}")

## Attribution: PF-GIM, GIM, EAP-IG

In [ ]:
results = {}

# --- GIM ---
print("=== GIM ===")
graph_gim = Graph.from_model(model)
attribute(model, graph_gim, dataloader, metric, method='GIM')
graph_gim.apply_topn(200, absolute=True)
score_gim = evaluate_graph(model, graph_gim, dataloader, metric, quiet=False).mean().item()
results['GIM'] = score_gim
print(f"GIM prob_diff: {score_gim:.4f}\n")

# --- EAP-IG ---
print("=== EAP-IG ===")
graph_eap = Graph.from_model(model)
attribute(model, graph_eap, dataloader, metric, method='EAP-IG-inputs', ig_steps=5)
graph_eap.apply_topn(200, absolute=True)
score_eap = evaluate_graph(model, graph_eap, dataloader, metric, quiet=False).mean().item()
results['EAP-IG'] = score_eap
print(f"EAP-IG prob_diff: {score_eap:.4f}\n")

# --- PF-GIM with different filter quantiles ---
for q in [0.1, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.7]:
    print(f"=== PF-GIM (q={q}) ===")
    graph_pf = Graph.from_model(model)
    attribute(model, graph_pf, dataloader, metric, method='PF-GIM',
              pf_gim_filter_quantile=q)
    graph_pf.apply_topn(200, absolute=True)
    score_pf = evaluate_graph(model, graph_pf, dataloader, metric, quiet=False).mean().item()
    results[f'PF-GIM (q={q})'] = score_pf
    print(f"PF-GIM (q={q}) prob_diff: {score_pf:.4f}\n")

## Summary

In [ ]:
print(f"{'Method':<25} {'prob_diff':>10} {'% of baseline':>15}")
print("-" * 52)
for name, score in results.items():
    pct = score / baseline_mean * 100 if baseline_mean != 0 else float('nan')
    print(f"{name:<25} {score:>10.4f} {pct:>14.1f}%")
print("-" * 52)
print(f"{'Full model':<25} {baseline_mean:>10.4f} {'100.0%':>15}")